In [1]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def read_and_merge_csv_files(folder_path):
    # List to hold individual DataFrames
    dataframes = []

    # Loop through all files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            # Read the CSV file into a DataFrame
            df = pd.read_csv(file_path, index_col=0)
            # Append the DataFrame to the list
            dataframes.append(df)

    # Concatenate all DataFrames in the list into a single DataFrame
    merged_df = pd.concat(dataframes, ignore_index=True)

    return merged_df
# Function to split the name column and create new columns
def split_name_column(name):
    parts = name.split('_')
    parameters = parts[-1].replace('.qasm', '').strip('[]')
    position = parts[5].replace('P', '')
    qubit = parts[6].replace('Q', '')
    return parts[0], parts[1], parts[3], parts[4], position, qubit, parameters


In [2]:
# Example usage:
folder_path = './results'
df = read_and_merge_csv_files(folder_path)
# Apply the function to the name column and create new columns
df[['Algorithm', 'Qubits_number',  'Operator', 'Gate', 'Position', 'Qubits', 'Params']] = df['Name'].apply(lambda x: pd.Series(split_name_column(x)))

# Drop the original name column if desired
df = df.drop(columns=['Name'])
df

,Input,Ideal_chisquare,Noisy_chisquare,Ideal_hellinger,Noisy_hellinger,Ideal_trace,Noisy_trace,Ideal_fidelity,Noisy_fidelity,Killed_IC,...,Killed_NT,Killed_IF,Killed_NF,Algorithm,Qubits_number,Operator,Gate,Position,Qubits,Params
0,PureState_0,1.408402e-208,2.225017e-252,0.267849,0.288319,0.336000,0.334381,0.500000,0.503281,True,...,True,True,True,ae,2,Add,rzz,2,0,1.5707963267948966
1,Quratest_0,7.407760e-118,1.390848e-128,0.244206,0.251714,0.291660,0.289583,0.557612,0.562213,True,...,True,True,True,ae,2,Add,rzz,2,0,1.5707963267948966
2,PureState_1,2.016730e-234,2.884031e-214,0.278582,0.269311,0.336000,0.334451,0.500000,0.503050,True,...,True,True,True,ae,2,Add,rzz,2,0,1.5707963267948966
3,Quratest_1,3.743449e-268,2.994642e-223,0.306515,0.282455,0.401266,0.398931,0.558245,0.562725,True,...,True,True,True,ae,2,Add,rzz,2,0,1.5707963267948966
4,PureState_2,2.884929e-208,4.452428e-241,0.266948,0.278188,0.336000,0.334705,0.500000,0.502973,True,...,True,True,True,ae,2,Add,rzz,2,0,1.5707963267948966
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22555,Quratest_29,0.000000e+00,0.000000e+00,0.568669,0.598379,0.635409,0.623003,0.006263,0.018192,True,...,True,True,True,wstate,6,Replace,rxx,12,2,3.9269908169872414
22556,PureState_30,0.000000e+00,0.000000e+00,0.888563,0.887644,0.853553,0.838829,0.048816,0.061676,True,...,True,True,True,wstate,6,Replace,rxx,12,2,3.9269908169872414
22557,Quratest_30,0.000000e+00,0.000000e+00,0.455622,0.435556,0.515817,0.498192,0.134129,0.157662,True,...,True,True,True,wstate,6,Replace,rxx,12,2,3.9269908169872414
22558,PureState_31,0.000000e+00,0.000000e+00,0.884655,0.856717,0.853553,0.838397,0.048816,0.061606,True,...,True,True,True,wstate,6,Replace,rxx,12,2,3.9269908169872414


In [3]:
# List of columns related to "Killed" metrics
killed_columns = [col for col in df.columns if col.startswith('Killed_')]

# Calculate the percentage of True values for each "Killed" column
true_percentages = (df[killed_columns].mean() * 100).sort_values(ascending=False)

# Print the results
print("Percentage of True values for each 'Killed' column:")
print(true_percentages)

Percentage of True values for each 'Killed' column:
Killed_NH    98.568262
Killed_NC    97.708333
Killed_NF    96.001773
Killed_NT    94.414894
Killed_IC    93.439716
Killed_IH    93.435284
Killed_IF    91.870567
Killed_IT    85.115248
dtype: float64


In [4]:
def confusionMatrix(df, col1, col2):
    # Create a confusion matrix DataFrame
    conf_matrix = pd.DataFrame(index=['True', 'False'], columns=['True', 'False'])
    
    # Calculate the count of each pair
    true_true = ((df[col1] == True) & (df[col2] == True)).sum()
    false_false = ((df[col1] == False) & (df[col2] == False)).sum()
    true_false = ((df[col1] == True) & (df[col2] == False)).sum()
    false_true = ((df[col1] == False) & (df[col2] == True)).sum()
    
    # Total number of rows
    total = len(df)
    
    # Calculate percentages
    conf_matrix.loc['True', 'True'] = (true_true / total) * 100
    conf_matrix.loc['False', 'False'] = (false_false / total) * 100
    conf_matrix.loc['True', 'False'] = (true_false / total) * 100
    conf_matrix.loc['False', 'True'] = (false_true / total) * 100
    
    # Ensure all values are numeric and handle any potential issues
    conf_matrix = conf_matrix.apply(pd.to_numeric, errors='coerce')  # Convert to numeric, coerce errors to NaN
    conf_matrix.fillna(0, inplace=True)  # Replace NaNs with 0 if there are any

    return conf_matrix

In [10]:
# Define a function to create a heatmap with annotations
def create_heatmap(fig, data, row, col, showscale):
    fig.add_trace(
        go.Heatmap(
            z=data,
            text=data,  # Use the same data for annotations
            colorscale='Dense',
            colorbar=dict(title='Scale'),
            zmin=0, zmax=100,
            showscale=showscale,
            texttemplate='%{text:.2f}',  # Format the text annotations
            textfont=dict(size=14)
        ),
        row=row, col=col
    )
    
    fig.update_xaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    fig.update_yaxes(tickvals=[0, 1], ticktext=['Killed', 'Survived'], row=row, col=col)
    

In [11]:

def printConfusionMatrixes(df_confusion, name):
    confusion_matrix_chisquare = confusionMatrix(df_confusion, 'Killed_IC','Killed_NC')
    confusion_matrix_hellinger = confusionMatrix(df_confusion, 'Killed_IH','Killed_NH')
    confusion_matrix_trace = confusionMatrix(df_confusion, 'Killed_IT','Killed_NT')
    confusion_matrix_fidelity = confusionMatrix(df_confusion, 'Killed_IF','Killed_NF')
    
    fig = make_subplots(rows=2, cols=2,
                        subplot_titles=('Chisquare', 'Hellinger', 'Trace', 'Fidelity'), x_title='Noisy', y_title='Ideal', horizontal_spacing=0.15,vertical_spacing=0.1)
    
    # Add heatmaps to subplots
    create_heatmap(fig, confusion_matrix_chisquare, row=1, col=1, showscale=True)
    create_heatmap(fig, confusion_matrix_hellinger, row=1, col=2, showscale=False)
    create_heatmap(fig, confusion_matrix_trace, row=2, col=1, showscale=False)
    create_heatmap(fig, confusion_matrix_fidelity, row=2, col=2, showscale=False)
    
    fig.update_layout(
        title_text=name,
        height=600,
        width=600,
        showlegend=False
    )
    
    fig.show()


In [12]:
# OVERALL CONFUSION MATRIX
printConfusionMatrixes(df, 'Overall confusion matrix')

In [13]:
# Confusion matrixes grouped by qubit numbers
qubit_numbers = df['Qubits_number'].unique()
for x in qubit_numbers:
    df_qubits = df[df['Qubits_number'] == x]
    printConfusionMatrixes(df_qubits, f'Confusion matrix for {x} qubits circuits')

In [14]:
# Confusion matrixes grouped by algorithms
algorithms = df['Algorithm'].unique()
for x in algorithms:
    df_algorithm = df[df['Algorithm'] == x]
    printConfusionMatrixes(df_algorithm, f'Confusion matrix for {x} algorithm')